# AI SkyRun · Tier 1+ — навчи дрон літати **з вітром**

Це **складніша версія** базового середовища. Дрон **не знає карти** та тепер також **не контролює кожний крок** — 5% часу вітер змінює напрямок на випадковий.

Це робить задачу справді **стохастичною**. Агент вчиться не заучувати шлях, а бути **стійким** до невизначеності.

> **Правило гри:** ви не можете читати `env_wind.trees`. Тільки досвід.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent.parent))

import numpy as np
import matplotlib.pyplot as plt

from tier_d.env_wind.gridworld import GridWorldWind
from tier_d.env.constants import N_ACTIONS
from tier_d.oracle.astar import optimal_length
from tier_d.scoring.seeds import PUBLIC_SEEDS

SEED, N_TREES = 0, 16
WIND_PROB = 0.05  # 5% вітру

env = GridWorldWind(N_TREES, SEED, wind_prob=WIND_PROB)
LSTAR = optimal_length(env.trees)  # оракул бачить дерева — ви ні

print(f"станів: {env.n_states}   дій: {env.n_actions}")
print(f"вітер: {100*WIND_PROB}% кроків перевизначаються")
print(f"оптимум оракула L* = {LSTAR:.3f}")

## Що відрізняється від базової версії?

| | Базова | З вітром |
|---|---|---|
| Перехід | Детерміністичний | 95% як задано, 5% випадково |
| Агент вчиться | Запам'ятовує шлях | Адаптується до невизначеності |
| Крива | Гладша, швидше | Шумніша, важче |
| L | Часто < L* | Часто > L* |
| Складність | Середня | **Висока** |

## Що дає середовище

Як в базовій версії, але `info` має додаткове поле:

```python
info["wind"]       # True якщо вітер змінив дію цього кроку
```

Решта ідентична: `collision`, `goal`, `truncated`, `moved`.

## TODO 1 — правило Беллмана (як в базовій версії)

У терміналі майбутнього немає: `target = r`.

In [ ]:
def q_update(Q, s, a, r, s2, terminal, alpha, gamma):
    """Оновити Q[s, a] на місці. Повернути TD-помилку (для дебагу)."""
    # TODO(1): як в базовій версії (вітер живе в середовищі, не у формулі)
    if terminal:
        target = r
    else:
        target = r + gamma * max(Q[s2])

    td = target - Q[s, a]
    Q[s, a] = Q[s, a] + alpha * td
    return td

## TODO 2 — reward shaping (як в базовій)

Розріджена винагорода буде ще більше дошкуляти. Shaping — обов'язково.

In [ ]:
from tier_d.env.constants import CELL, GRID_N
from tier_d.env.occupancy import GOAL_CELL

USE_SHAPING = True  # Ви **будете** потребувати цього

def potential(s):
    r, c = divmod(s, GRID_N)
    gr, gc = GOAL_CELL
    return -np.hypot(r - gr, c - gc) * CELL

def shaping_F(s, s2, gamma, terminal):
    if not USE_SHAPING:
        return 0.0
    phi_next = 0.0 if terminal else potential(s2)
    return gamma * phi_next - potential(s)

## TODO 3 — гіперпараметри (потребуватимуть іншого тюнінгу!)

З вітром агент тренується важче. Попробуйте більші значення EPISODES й можливо інший decay.

In [ ]:
ALPHA   = 0.05      # тюнінг під вітер: менший темп → стабільніше під стохастикою
GAMMA   = 0.99      # Залиште як є
EPS0    = 1.0
EPS_MIN = 0.01      # менше випадкових детурів наприкінці → 100% успіху
EPISODES = 5000     # більше ніж базова версія (3000) → тренування довше

## Тренування

In [ ]:
def greedy_rollout(env, Q, max_steps=400):
    s = env.reset(); path = [env.xy]; length = 0.0
    for _ in range(max_steps):
        s, _, done, info = env.step(int(np.argmax(Q[s])))
        path.append(env.xy); length += info["moved"]
        if info["collision"]: return dict(success=False, length=length, path=np.array(path))
        if info["goal"]:      return dict(success=True,  length=length, path=np.array(path))
    return dict(success=False, length=length, path=np.array(path))


def train(env, episodes=EPISODES, seed=0):
    rng = np.random.default_rng(seed)
    Q = np.zeros((env.n_states, N_ACTIONS))
    curve = []
    decay = int(episodes * 0.6)

    for ep in range(episodes):
        eps = max(EPS_MIN, EPS0 + (EPS_MIN - EPS0) * ep / decay)
        s = env.reset()
        while True:
            a = int(rng.integers(N_ACTIONS)) if rng.random() < eps else int(np.argmax(Q[s]))
            s2, r, done, info = env.step(a)
            terminal = info["collision"] or info["goal"]

            r_train = r + shaping_F(s, s2, GAMMA, terminal)
            q_update(Q, s, a, r_train, s2, terminal, ALPHA, GAMMA)

            s = s2
            if done: break

        if ep % 25 == 0:
            roll = greedy_rollout(env, Q)
            curve.append((ep, roll["length"] if roll["success"] else np.nan))
    return Q, np.array(curve)


Q, curve = train(GridWorldWind(N_TREES, SEED, wind_prob=WIND_PROB))
roll = greedy_rollout(GridWorldWind(N_TREES, SEED, wind_prob=WIND_PROB), Q)
print(f"дійшов: {roll['success']}   L = {roll['length']:.3f}   L* = {LSTAR:.3f}")
if roll["success"]:
    print(f"оптимальність L*/L = {LSTAR / roll['length']:.3f}")

## Крива навчання

Очікуйте більш шумну криву через вітер.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.axhline(LSTAR, ls="--", color="#8250df", label=f"L* = {LSTAR:.2f}")
ax.axhline(1.15 * LSTAR, ls=":", color="grey", label="поріг 1.15·L*")
ok = ~np.isnan(curve[:, 1])
ax.plot(curve[ok, 0], curve[ok, 1], color="#d1260d", lw=1.7, label="жадібна політика (з вітром)")
ax.set_xlabel("епізод"); ax.set_ylabel("довжина шляху"); ax.legend(); ax.grid(alpha=.3)
plt.show()

## Порівняння з базовою версією

```bash
python tier_d/experiments/compare_wind.py --seed 0
```

Це створить графік, де видна різниця між детерміністичною й стохастичною версіями.

## Самоперевірка на public seeds

```bash
python tier_d/scaffold/validate_team.py
```